# Module 5.6 — Distributed Task Concepts

### Python Mastery · Track 5: Concurrency & Asynchronous Programming

When work outgrows a single program, it is distributed across many workers and often many machines. Real systems use task queues such as Celery or RQ, message brokers, and databases. This module teaches the **concepts** that make such systems reliable, illustrated with small, self-contained simulations you can run and reason about, without installing any infrastructure.

**How to use this notebook**

- Read each explanation, then run the simulation beneath it. The simulations model the behaviour of real systems in plain Python so the ideas are concrete.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Describe the task-queue model: producers, a broker, and workers.
2. Explain idempotency and why distributed tasks need it.
3. Implement retries with exponential backoff.
4. Describe dead-letter queues for handling repeated failures.
5. Reason about eventual consistency and at-least-once delivery.

**Prerequisites:** Tracks 1 to 4, and Modules 5.1 to 5.5.

---

## Part 1 · The Task-Queue Model

Distributed task systems separate the act of **requesting** work from **doing** it. A producer puts a task description onto a queue (held by a **broker** such as Redis or RabbitMQ). One or more **workers**, possibly on different machines, take tasks from the queue and execute them. This decouples components, smooths bursts of load, and lets you scale by adding workers.

The simulation below models this with an in-memory queue and a worker loop, capturing the shape of the real pattern.

In [ ]:
from collections import deque

# A broker is, in essence, a shared queue of pending tasks.
broker = deque()

def enqueue(task):
    """A producer adds a task (a description of work to do)."""
    broker.append(task)

def worker(broker):
    """A worker repeatedly takes the next task and processes it."""
    done = []
    while broker:
        task = broker.popleft()                 # take the next task
        result = task["value"] ** 2             # 'do the work'
        done.append((task["id"], result))
    return done

# Producers enqueue several tasks.
for i in range(1, 6):
    enqueue({"id": f"task-{i}", "value": i})

print("processed:", worker(broker))

## Part 2 · Idempotency

In a distributed system, a task may be delivered and run **more than once**: a worker might crash after doing the work but before acknowledging it, so the task is retried. An operation is **idempotent** if running it several times has the same effect as running it once. Designing tasks to be idempotent is the key defence against duplicates.

For example, "set the account status to active" is idempotent; "add 10 to the balance" is not. The simulation shows how recording processed task identifiers makes a non-idempotent action safe.

In [ ]:
# A non-idempotent action made safe by tracking which tasks have been applied.
balance = 0
processed_ids = set()

def apply_credit(task):
    global balance
    if task["id"] in processed_ids:
        return "skipped (already applied)"      # the guard makes re-delivery harmless
    balance += task["amount"]
    processed_ids.add(task["id"])
    return "applied"

credit = {"id": "credit-001", "amount": 100}

print("first delivery:", apply_credit(credit))
print("duplicate delivery:", apply_credit(credit))   # safely ignored
print("final balance:", balance)                      # 100, not 200

## Part 3 · Retries with Exponential Backoff

Tasks fail for transient reasons: a network blip, a momentarily overloaded service. Retrying often succeeds, but retrying immediately and repeatedly can overwhelm a struggling system. **Exponential backoff** waits longer after each failure (for example 1, 2, 4, 8 units), giving the system time to recover. Adding small random **jitter** avoids many clients retrying in lockstep.

In [ ]:
def with_retries(action, max_attempts=5, base_delay=1):
    """Call action(); on failure, retry with exponentially growing (simulated) delays."""
    delays_used = []
    for attempt in range(max_attempts):
        try:
            return action(attempt), delays_used
        except Exception as e:
            if attempt == max_attempts - 1:
                raise                              # give up after the last attempt
            delay = base_delay * (2 ** attempt)    # 1, 2, 4, 8, ...
            delays_used.append(delay)
            # In a real system you would sleep here; we only record the delay.

# An action that fails the first three times, then succeeds.
def flaky(attempt):
    if attempt < 3:
        raise ConnectionError("temporary failure")
    return "success"

result, delays = with_retries(flaky)
print("result:", result)
print("backoff delays used between attempts:", delays)   # [1, 2, 4]

## Part 4 · Dead-Letter Queues

Some tasks fail no matter how many times they are retried, perhaps the input is malformed. Retrying them forever wastes resources and can block the queue. A **dead-letter queue** is a separate place where tasks are sent after exhausting their retries, so they can be inspected and handled later without holding up healthy work.

In [ ]:
def process_with_dead_letter(tasks, max_attempts=3):
    completed = []
    dead_letter = []                               # holds tasks that never succeed

    def attempt(task):
        # 'poison' tasks always fail; others succeed.
        if task["kind"] == "poison":
            raise ValueError("cannot process")
        return task["id"]

    for task in tasks:
        for tries in range(1, max_attempts + 1):
            try:
                completed.append(attempt(task))
                break
            except ValueError:
                if tries == max_attempts:
                    dead_letter.append(task["id"]) # park it after final failure
    return completed, dead_letter

tasks = [
    {"id": "ok-1", "kind": "normal"},
    {"id": "bad-1", "kind": "poison"},
    {"id": "ok-2", "kind": "normal"},
    {"id": "bad-2", "kind": "poison"},
]

completed, dead = process_with_dead_letter(tasks)
print("completed:", completed)
print("dead-letter queue:", dead)

## Part 5 · Delivery Guarantees and Eventual Consistency

Distributed systems make trade-offs about **delivery guarantees**. Most practical systems offer **at-least-once** delivery: a task will run, but possibly more than once (hence the need for idempotency). Exactly-once is very hard and usually approximated by at-least-once plus idempotency.

Closely related is **eventual consistency**: after a change, different parts of a system may briefly disagree, but given time and no new changes they converge to the same state. Designing for this means tolerating temporary staleness rather than assuming every read is instantly up to date. The simulation models replicas converging after an update propagates.

In [ ]:
# Model two replicas that converge after an update propagates.
class Replica:
    def __init__(self, name):
        self.name = name
        self.value = None

primary = Replica("primary")
secondary = Replica("secondary")

# An update hits the primary first.
primary.value = "v2"
print("immediately after write:")
print(f"  {primary.name}={primary.value}, {secondary.name}={secondary.value}")
print("  -> replicas temporarily disagree (this is normal)")

# Propagation happens a moment later (here, explicitly).
def propagate(source, target):
    target.value = source.value

propagate(primary, secondary)
print("after propagation:")
print(f"  {primary.name}={primary.value}, {secondary.name}={secondary.value}")
print("  -> the system has become consistent")

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — A minimal queue and worker (Easy)

In [ ]:
from collections import deque

queue = deque(["job1", "job2", "job3"])
processed = []
while queue:
    job = queue.popleft()
    processed.append(job.upper())
print("processed in order:", processed)

### Example 2 — Guarding against duplicate processing (Easy)

In [ ]:
seen = set()
results = []

def handle(task_id):
    if task_id in seen:
        return
    seen.add(task_id)
    results.append(task_id)

for tid in ["a", "b", "a", "c", "b"]:    # duplicates present
    handle(tid)
print("handled once each:", results)

### Example 3 — Computing a backoff schedule (Medium)

In [ ]:
def backoff_schedule(attempts, base=1, cap=30):
    """Return the delay before each retry, capped at a maximum."""
    return [min(cap, base * (2 ** i)) for i in range(attempts)]

print("delays (base 1):", backoff_schedule(6))        # [1, 2, 4, 8, 16, 30]
print("delays (base 2, cap 20):", backoff_schedule(5, base=2, cap=20))

### Example 4 — Retry until success or give up (Medium)

In [ ]:
def run_with_retry(action, attempts=4):
    for i in range(attempts):
        outcome = action(i)
        if outcome != "fail":
            return f"succeeded on attempt {i + 1}"
    return "gave up"

# Fails twice, then succeeds.
def task(attempt):
    return "fail" if attempt < 2 else "ok"

print(run_with_retry(task))

### Example 5 — A worker with retries and a dead-letter queue (Difficult)

In [ ]:
def worker(tasks, max_attempts=3):
    completed, dead_letter = [], []

    def do_work(task, attempt):
        # 'transient' tasks fail until their final attempt; 'broken' always fail.
        if task["kind"] == "broken":
            raise ValueError("permanent failure")
        if task["kind"] == "transient" and attempt < 2:
            raise ConnectionError("temporary failure")
        return task["id"]

    for task in tasks:
        for attempt in range(max_attempts):
            try:
                completed.append(do_work(task, attempt))
                break
            except Exception:
                if attempt == max_attempts - 1:
                    dead_letter.append(task["id"])
    return completed, dead_letter

tasks = [
    {"id": "A", "kind": "normal"},
    {"id": "B", "kind": "transient"},
    {"id": "C", "kind": "broken"},
]
completed, dead = worker(tasks)
print("completed:", completed)
print("dead-letter:", dead)

### Example 6 — Simulating at-least-once delivery with idempotent handling (Difficult)

In [ ]:
import random
random.seed(0)                               # deterministic for reproducibility

# A task may be delivered more than once (at-least-once). Idempotent handling keeps
# the result correct regardless of duplicates.
applied = set()
account = {"points": 0}

def deliver(task, times):
    """Simulate the broker delivering a task 'times' times."""
    for _ in range(times):
        if task["id"] not in applied:
            account["points"] += task["points"]
            applied.add(task["id"])

# Each task is delivered a random number of times (1 to 3), but applied once.
tasks = [{"id": f"t{i}", "points": 10} for i in range(5)]
for task in tasks:
    deliver(task, random.randint(1, 3))

print("tasks delivered with duplicates, but points credited once each")
print("total points:", account["points"])   # exactly 5 * 10 = 50

---

## Exercises

**Exercise 1 (Easy).** Using a `deque` as a queue, enqueue the strings `"a"`, `"b"`, `"c"`, then process them in order, collecting each in uppercase. Print the result.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Write a function `handle(task_id)` that processes a task only once, using a set to remember processed identifiers. Feed it a list with duplicates and print the list of uniquely processed identifiers.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Write a function `backoff(n)` returning the list of exponential delays `[1, 2, 4, ...]` for `n` attempts, capped at 60. Print the schedule for 8 attempts.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Write a retry loop that calls an action up to 5 times; the action fails (returns `"fail"`) for the first three calls and then succeeds. Return how many attempts it took.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Simulate a worker that processes a list of tasks, retrying each up to 3 times. Tasks marked `"poison"` always fail and should end in a dead-letter list; others succeed immediately. Print the completed and dead-letter lists.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
from collections import deque
queue = deque(["a", "b", "c"])
done = []
while queue:
    done.append(queue.popleft().upper())
print(done)

**Solution 2**

In [ ]:
seen = set()
processed = []

def handle(task_id):
    if task_id not in seen:
        seen.add(task_id)
        processed.append(task_id)

for tid in ["x", "y", "x", "z", "y", "z"]:
    handle(tid)
print(processed)

**Solution 3**

In [ ]:
def backoff(n, cap=60):
    return [min(cap, 2 ** i) for i in range(n)]

print(backoff(8))

**Solution 4**

In [ ]:
def run(action, attempts=5):
    for i in range(attempts):
        if action(i) != "fail":
            return i + 1
    return None

def action(attempt):
    return "fail" if attempt < 3 else "ok"

print("succeeded after", run(action), "attempts")

**Solution 5**

In [ ]:
def worker(tasks, max_attempts=3):
    completed, dead = [], []
    for task in tasks:
        for attempt in range(max_attempts):
            try:
                if task["kind"] == "poison":
                    raise ValueError("always fails")
                completed.append(task["id"])
                break
            except ValueError:
                if attempt == max_attempts - 1:
                    dead.append(task["id"])
    return completed, dead

tasks = [
    {"id": "1", "kind": "normal"},
    {"id": "2", "kind": "poison"},
    {"id": "3", "kind": "normal"},
]
print(worker(tasks))

---

## Summary and Key Points

- Task queues separate requesting work from doing it: producers enqueue tasks to a broker, and workers (possibly across machines) consume and execute them, which decouples components and scales by adding workers.
- Tasks may run more than once, so design them to be idempotent, meaning repeated execution has the same effect as a single execution; tracking processed identifiers makes non-idempotent actions safe.
- Transient failures are handled by retries with exponential backoff (growing delays, often with jitter) to avoid overwhelming a recovering system.
- Tasks that fail repeatedly are moved to a dead-letter queue for later inspection, so they do not block healthy work.
- Practical systems offer at-least-once delivery and eventual consistency: parts may briefly disagree but converge, so tolerate temporary staleness and pair at-least-once delivery with idempotency.

### Next module: 5.7 — Mixing Sync & Async

The next module covers bridging the synchronous and asynchronous worlds: running blocking code from async with `to_thread` and executors, and avoiding blocking the event loop.